In [ ]:
#!/usr/bin/env python3
"""Reduce PV nowcasting MAE/RMSE with validation-selected LightGBM models.

The script uses the uploaded OPERA 10-minute and 90-minute engineered-feature CSVs.
It does NOT use Pa1, electrical output current, or voltage as input. Optional engineered
predictors are timestamp-derived calendar features and interactions from environmental sensors.
Companion CSVs summarize prior Random Forest/XGBoost reference results on fixed test splits.

Default protocol: shuffled 80/20 holdout, with a separate validation subset from the
outer training data. Use --chronological for a separate forward-in-time test.

Install:
  python -m pip install lightgbm pandas numpy scikit-learn joblib

Run:
  python pv_lightgbm_error_reduction.py
  python pv_lightgbm_error_reduction.py --chronological
"""
from __future__ import annotations
import argparse, json, os, sys, time, warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

SEED = 42
TARGET = "Pa1"

# Fallback to current working directory in interactive/notebook environments where __file__ is undefined
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

DATA_10M = ROOT / "opera_dataset_1w_10min.csv"
DATA_90M = ROOT / "opera_dataset_3w_90min.csv"
OUT_DEFAULT = ROOT / "pv_lightgbm_results"
N_JOBS = min(4, os.cpu_count() or 1)
PAPER = {"r2": 0.9983, "mae_w": 173.47, "rmse_w": 360.13}

# Environmental-only configurations: no target-derived values.
RANDOM_CONFIGS = {
    "LGBM_RMSE_31": dict(objective="regression", n_estimators=1800, learning_rate=0.025,
        num_leaves=31, max_depth=-1, min_child_samples=25, subsample=0.9, subsample_freq=1,
        colsample_bytree=0.9, reg_lambda=3.0, reg_alpha=0.05),
    "LGBM_RMSE_63": dict(objective="regression", n_estimators=1600, learning_rate=0.025,
        num_leaves=63, max_depth=-1, min_child_samples=40, subsample=0.9, subsample_freq=1,
        colsample_bytree=0.9, reg_lambda=5.0, reg_alpha=0.1),
    "LGBM_RMSE_127": dict(objective="regression", n_estimators=1400, learning_rate=0.03,
        num_leaves=127, max_depth=-1, min_child_samples=60, subsample=0.85, subsample_freq=1,
        colsample_bytree=0.9, reg_lambda=8.0, reg_alpha=0.1),
    "LGBM_MAE_31": dict(objective="regression_l1", n_estimators=1800, learning_rate=0.025,
        num_leaves=31, max_depth=-1, min_child_samples=25, subsample=0.9, subsample_freq=1,
        colsample_bytree=0.9, reg_lambda=3.0, reg_alpha=0.05),
}
CHRONO_CONFIGS = {
    "LGBM_RMSE_31": {**RANDOM_CONFIGS["LGBM_RMSE_31"], "n_estimators": 2400},
    "LGBM_RMSE_63": {**RANDOM_CONFIGS["LGBM_RMSE_63"], "n_estimators": 1800},
    "LGBM_RMSE_127": {**RANDOM_CONFIGS["LGBM_RMSE_127"], "n_estimators": 1600},
    "LGBM_MAE_31": {**RANDOM_CONFIGS["LGBM_MAE_31"], "n_estimators": 2400},
}

def metric_dict(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae_w": float(mean_absolute_error(y_true, y_pred)),
        "rmse_w": float(mean_squared_error(y_true, y_pred) ** 0.5),
    }


def selection_score(metrics):
    # Optimize errors jointly; MAE has slightly greater weight.
    return 0.55 * metrics["mae_w"] / PAPER["mae_w"] + 0.45 * metrics["rmse_w"] / PAPER["rmse_w"]


def load_aligned_data(path_10m=DATA_10M, path_90m=DATA_90M):
    """Keep 90-min samples whose matching 10-min stream has a complete trailing 9 steps."""
    for path in (path_10m, path_90m):
        if not path.exists():
            raise FileNotFoundError(f"Missing {path.name}; put it beside this script or pass the proper path.")
    base = pd.read_csv(path_10m)
    data = pd.read_csv(path_90m)
    for frame in (base, data):
        if "timestamp" not in frame or TARGET not in frame:
            raise ValueError(f"Each CSV must contain timestamp and {TARGET} columns.")
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True, errors="coerce")
    base = base.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    data = data.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    if base.timestamp.duplicated().any() or data.timestamp.duplicated().any():
        raise ValueError("Duplicate timestamps found; inspect source files rather than silently deduplicating.")

    base_ts = pd.DatetimeIndex(base.timestamp)
    base_pos = {ts: i for i, ts in enumerate(base_ts)}
    ten_min_ns = pd.Timedelta(minutes=10).value
    keep = []
    for i, ts in enumerate(data.timestamp):
        j = base_pos.get(ts)
        if j is None or j < 8:
            continue
        window_idx = np.arange(j - 8, j + 1)
        if np.all(np.diff(base_ts.asi8[window_idx]) == ten_min_ns):
            keep.append(i)
    data = data.iloc[keep].reset_index(drop=True)
    if data.empty:
        raise ValueError("No aligned rows remain; confirm that both CSVs come from the same source series.")

    overlap = base[["timestamp", TARGET]].merge(
        data[["timestamp", TARGET]], on="timestamp", suffixes=("_10m", "_90m")
    )
    if len(overlap) and not np.allclose(
        overlap[f"{TARGET}_10m"].to_numpy(float), overlap[f"{TARGET}_90m"].to_numpy(float),
        rtol=0, atol=1e-6
    ):
        raise ValueError("Pa1 differs between the 10-minute and 90-minute files.")

    features = [c for c in data.columns if c not in ("timestamp", TARGET)]
    forbidden = [c for c in features if c.lower() in {"pa1", "power", "current", "voltage", "ia", "va"}]
    if forbidden:
        raise ValueError(f"Potential target leakage in predictors: {forbidden}")
    X = data[features].apply(pd.to_numeric, errors="coerce")
    if X.isna().any().any() or data[TARGET].isna().any():
        raise ValueError("Missing/non-numeric features or targets found; clean and document them explicitly.")
    return data, features, X.astype(np.float32), data[TARGET].to_numpy(dtype=np.float64)


def engineered_features(data, X_raw):
    """Known-at-prediction-time calendar features and environmental interactions."""
    X = X_raw.copy()
    ts = pd.to_datetime(data.timestamp, utc=True)
    minute_of_day = (ts.dt.hour * 60 + ts.dt.minute).to_numpy()
    day_of_year = ts.dt.dayofyear.to_numpy()
    X["time_sin"] = np.sin(2 * np.pi * minute_of_day / 1440.0)
    X["time_cos"] = np.cos(2 * np.pi * minute_of_day / 1440.0)
    X["day_of_year_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    X["day_of_year_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
    X["rad_avg_squared"] = X["Rad_avg"] ** 2
    X["rad_module_temp_interaction"] = X["Rad_avg"] * X["Tmod_avg"]
    X["rad_ambient_temp_interaction"] = X["Rad_avg"] * X["Tamb_avg"]
    X["module_minus_ambient_temp"] = X["Tmod_avg"] - X["Tamb_avg"]
    X["rad_change_w2"] = X["Rad_avg"] - X["Rad_avg_w2"]
    X["rad_change_w3"] = X["Rad_avg"] - X["Rad_avg_w3"]
    return X.astype(np.float32)


def fit_lgb_validation(X_train, y_train, X_val, y_val, cfg, patience):
    model = LGBMRegressor(**cfg, random_state=SEED, n_jobs=N_JOBS, verbosity=-1)
    start = time.perf_counter()
    model.fit(
        X_train, y_train, eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(patience, verbose=False), log_evaluation(0)]
    )
    fit_seconds = time.perf_counter() - start
    pred = model.predict(X_val)
    return model, pred, metric_dict(y_val, pred), fit_seconds


def select_single_or_blend(candidates, val_predictions, y_val, allow_blend=True):
    """Selection is exclusively validation-based; no test labels are passed here."""
    single_rows = []
    for row in candidates:
        m = {"r2": row["val_r2"], "mae_w": row["val_mae_w"], "rmse_w": row["val_rmse_w"]}
        single_rows.append({"kind": "single", "name": row["name"], "selection_score": selection_score(m)})
    best_single = min(single_rows, key=lambda r: r["selection_score"])

    # Restrict blending to the five lowest-validation-MAE models to keep the search compact.
    top_names = [r["name"] for r in sorted(candidates, key=lambda z: z["val_mae_w"])[:5]]
    blend_rows = []
    for i, name_a in enumerate(top_names):
        for name_b in top_names[i + 1:]:
            for weight_a in (0.2, 0.35, 0.5, 0.65, 0.8):
                pred = weight_a * val_predictions[name_a] + (1 - weight_a) * val_predictions[name_b]
                m = metric_dict(y_val, pred)
                blend_rows.append({
                    "kind": "blend", "name": f"{name_a}__{weight_a:.2f}__{name_b}",
                    "member_a": name_a, "member_b": name_b, "weight_a": weight_a,
                    **{f"val_{k}": v for k, v in m.items()}, "selection_score": selection_score(m)
                })
    best_blend = min(blend_rows, key=lambda r: r["selection_score"]) if blend_rows else None
    if allow_blend and best_blend and best_blend["selection_score"] < best_single["selection_score"]:
        return best_blend, pd.DataFrame(blend_rows).sort_values("selection_score")
    return best_single, pd.DataFrame(blend_rows).sort_values("selection_score")


def run_protocol(data, raw_features, X_raw, y, output_dir, chronological=False):
    output_dir.mkdir(parents=True, exist_ok=True)
    if chronological:
        n = len(y)
        cut1, cut2, gap = int(0.70 * n), int(0.85 * n), 9
        train_idx = np.arange(cut1)
        val_idx = np.arange(min(n, cut1 + gap), cut2)
        test_idx = np.arange(min(n, cut2 + gap), n)
        patience = 100
        configs = {k: CHRONO_CONFIGS[k] for k in ("LGBM_RMSE_127", "LGBM_MAE_31")}
        protocol = "chronological 70/15/15; 9 samples purged at train/validation and validation/test boundaries"
    else:
        idx = np.arange(len(y))
        train_idx, test_idx = train_test_split(idx, test_size=0.20, random_state=SEED, shuffle=True)
        fit_idx, val_idx = train_test_split(train_idx, test_size=0.20, random_state=43, shuffle=True)
        train_idx_for_fit = fit_idx
        patience = 80
        configs = RANDOM_CONFIGS
        protocol = "random 80/20 holdout with a separate validation subset within outer training data"

    if chronological:
        variants = [("engineered", engineered_features(data, X_raw))]
    else:
        variants = [("env_only", X_raw), ("engineered", engineered_features(data, X_raw))]
    candidates, val_predictions, models_meta = [], {}, {}
    print(f"\nProtocol: {protocol}", flush=True)
    print(f"Usable data rows: {len(data):,}; target={TARGET}; input features={len(raw_features)} environmental features", flush=True)
    for schema, frame in variants:
        X = frame.to_numpy(dtype=np.float32)
        train_fit = train_idx if chronological else train_idx_for_fit
        for name, cfg in configs.items():
            full_name = f"{schema}__{name}"
            print(f"Validation fit: {full_name}", flush=True)
            m, vp, vm, sec = fit_lgb_validation(X[train_fit], y[train_fit], X[val_idx], y[val_idx], cfg, patience)
            best_iter = int(getattr(m, "best_iteration_", 0) or cfg["n_estimators"])
            row = {
                "name": full_name, "schema": schema, "config_name": name,
                "config": json.dumps(cfg, sort_keys=True), "best_iteration": best_iter,
                "validation_fit_seconds": sec,
                **{f"val_{k}": v for k, v in vm.items()},
                "selection_score": selection_score(vm),
            }
            candidates.append(row)
            val_predictions[full_name] = vp
            models_meta[full_name] = {"schema": schema, "config_name": name, "config": cfg.copy(), "best_iteration": best_iter,
                                      "features": frame.columns.tolist()}
            print(f"  MAE={vm['mae_w']:.2f} W | RMSE={vm['rmse_w']:.2f} W | R2={vm['r2']:.6f} | best_iter={best_iter}", flush=True)

    val_table = pd.DataFrame(candidates).sort_values("selection_score")
    suffix = "chronological" if chronological else "random"
    val_table.to_csv(output_dir / f"{suffix}_validation_search.csv", index=False)
    chosen, blend_table = select_single_or_blend(candidates, val_predictions, y[val_idx], allow_blend=chronological)
    blend_table.to_csv(output_dir / f"{suffix}_validation_blend_search.csv", index=False)
    print(f"Validation-selected final choice: {chosen['kind']} / {chosen['name']}", flush=True)

    if chosen["kind"] == "blend":
        selected_names = [chosen["member_a"], chosen["member_b"]]
    else:
        selected_names = [chosen["name"]]

    X_lookup = {schema: frame.to_numpy(dtype=np.float32) for schema, frame in variants}
    fitted, test_predictions, final_rows = {}, {}, []
    for name in selected_names:
        meta = models_meta[name]
        cfg = meta["config"].copy()
        cfg["n_estimators"] = meta["best_iteration"]
        X = X_lookup[meta["schema"]]
        model = LGBMRegressor(**cfg, random_state=SEED, n_jobs=N_JOBS, verbosity=-1)
        tic = time.perf_counter()
        # Fit on the outer training period/samples only; validation is never included in training.
        model.fit(X[train_idx], y[train_idx])
        fit_sec = time.perf_counter() - tic
        pred = model.predict(X[test_idx])
        test_predictions[name] = pred
        fitted[name] = model
        mt = metric_dict(y[test_idx], pred)
        final_rows.append({"model": name, "schema": meta["schema"], "config_name": meta["config_name"],
                           "fit_seconds": fit_sec, **mt})
        joblib.dump({"model": model, "features": meta["features"], "target": TARGET,
                     "model_name": name, "configuration": cfg},
                    output_dir / f"{suffix}_{name}.joblib", compress=3)

    if chosen["kind"] == "blend":
        wa = float(chosen["weight_a"])
        final_pred = wa * test_predictions[chosen["member_a"]] + (1 - wa) * test_predictions[chosen["member_b"]]
        final_name = f"Selected_LightGBM_blend_{wa:.2f}_{1-wa:.2f}"
    else:
        final_pred = test_predictions[chosen["name"]]
        final_name = "Selected_LightGBM_single"

    final_rows.append({"model": final_name, "schema": "validation-selected single/blend",
                       "config_name": chosen["name"], "fit_seconds": sum(r["fit_seconds"] for r in final_rows),
                       **metric_dict(y[test_idx], final_pred)})

    result = pd.DataFrame(final_rows).drop_duplicates(subset=["model"], keep="last")
    result["mae_reduction_vs_paper_pct"] = 100 * (PAPER["mae_w"] - result.mae_w) / PAPER["mae_w"]
    result["rmse_reduction_vs_paper_pct"] = 100 * (PAPER["rmse_w"] - result.rmse_w) / PAPER["rmse_w"]
    result.to_csv(output_dir / f"{suffix}_final_comparison.csv", index=False)

    pred_df = pd.DataFrame({
        "timestamp": data.loc[test_idx, "timestamp"].astype(str).to_numpy(),
        "actual_power_w": y[test_idx], "selected_model_prediction_w": final_pred,
    })
    for name, pred in test_predictions.items():
        pred_df[f"{name}_prediction_w"] = pred
    pred_df.to_csv(output_dir / f"{suffix}_test_predictions.csv", index=False)

    recipe = {
        "protocol": protocol, "selected_model": chosen, "test_rows": len(test_idx),
        "training_rows": len(train_idx), "validation_rows": len(val_idx),
        "test_start": str(data.loc[test_idx[0], "timestamp"]) if chronological else None,
        "paper_reference": PAPER,
        "important_note": (
            "The paper's reported numbers use a different cross-validation protocol. Random holdout results are not directly protocol-matched. "
            "Calendar features are known at inference time, but mean this is a model+feature-engineering comparison, not an architecture-only comparison. "
            "Chronological results are the stronger check of generalization to later periods."
        ),
    }
    (output_dir / f"{suffix}_selected_model_recipe.json").write_text(json.dumps(recipe, indent=2, default=str), encoding="utf-8")

    show_cols = ["model", "r2", "mae_w", "rmse_w"]
    print("\nFINAL TEST COMPARISON (same test rows)", flush=True)
    print(result[show_cols].to_string(index=False), flush=True)
    print(f"Saved tables, predictions, model artifacts and recipe to: {output_dir}", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--chronological", action="store_true", help="Run a strictly time-ordered 70/15/15 evaluation instead of random holdout.")
    parser.add_argument("--data-10m", type=Path, default=DATA_10M)
    parser.add_argument("--data-90m", type=Path, default=DATA_90M)
    parser.add_argument("--out", type=Path, default=OUT_DEFAULT)
    args = parser.parse_args()
    warnings.filterwarnings("ignore", category=UserWarning)
    print("Loading aligned OPERA dataset...", flush=True)
    data, raw_features, X_raw, y = load_aligned_data(args.data_10m, args.data_90m)
    run_protocol(data, raw_features, X_raw, y, args.out, chronological=args.chronological)


if __name__ == "__main__":
    # In interactive notebook, main() can be executed directly with default settings
    if 'ipykernel' in sys.modules:
        warnings.filterwarnings("ignore", category=UserWarning)
        print("Running main() directly in interactive environment...", flush=True)
        data, raw_features, X_raw, y = load_aligned_data(DATA_10M, DATA_90M)
        run_protocol(data, raw_features, X_raw, y, OUT_DEFAULT, chronological=False)
    else:
        main()

Running main() directly in interactive environment...

Protocol: random 80/20 holdout with a separate validation subset within outer training data
Usable data rows: 23,984; target=Pa1; input features=36 environmental features
Validation fit: env_only__LGBM_RMSE_31
  MAE=146.35 W | RMSE=311.22 W | R2=0.998727 | best_iter=1800
Validation fit: env_only__LGBM_RMSE_63
  MAE=151.46 W | RMSE=319.20 W | R2=0.998661 | best_iter=470
Validation fit: env_only__LGBM_RMSE_127
  MAE=148.69 W | RMSE=321.54 W | R2=0.998641 | best_iter=522
Validation fit: env_only__LGBM_MAE_31
  MAE=151.94 W | RMSE=350.76 W | R2=0.998383 | best_iter=1800
Validation fit: engineered__LGBM_RMSE_31
  MAE=107.40 W | RMSE=264.75 W | R2=0.999079 | best_iter=652
Validation fit: engineered__LGBM_RMSE_63
  MAE=102.39 W | RMSE=265.91 W | R2=0.999071 | best_iter=472
Validation fit: engineered__LGBM_RMSE_127
  MAE=99.47 W | RMSE=266.92 W | R2=0.999064 | best_iter=505
Validation fit: engineered__LGBM_MAE_31
  MAE=107.24 W | RMSE=284.